# 04 — Embedding Generation

In [1]:
import sys
sys.path.insert(0, "..")
from src import config, data_loader, embeddings as emb_mod

config.ensure_directories()
df = data_loader.load_final_dataset()
col_map = data_loader.get_available_columns(df)
texts = df["search_text"].tolist()
print(f"{len(texts)} documents to embed")

backend = emb_mod.resolve_backend()
print("Embedding backend in use:", backend)


2026-09-01 22:03:08,161 | INFO     | src.data_loader | Loaded final dataset: 583 rows from E:\Job Base Programe\ResearchMind\data\processed\research_papers_final.csv


583 documents to embed
Embedding backend in use: sentence-transformers


## Generate embeddings (batched)

In [2]:
vectors = emb_mod.embed_documents(texts, batch_size=config.EMBEDDING_BATCH_SIZE, backend=backend)
print("Embeddings shape:", vectors.shape)
assert vectors.shape[0] == len(df), "Embedding count must match document count."


2026-09-01 22:06:09,086 | INFO     | src.embeddings | Loading sentence-transformers model 'all-mpnet-base-v2' (this may download weights on first use)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\AC\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\AC\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Embeddings shape: (583, 768)


## Build index -> metadata mapping

Every embedding row must map 1:1 to a metadata record.

In [3]:
metadata = []
for idx, row in df.iterrows():
    metadata.append({
        "index": int(idx),
        "paper_id": row.get(col_map.get("paper_id"), ""),
        "title": row.get(col_map.get("title"), ""),
        "authors": row.get(col_map.get("authors"), ""),
        "year": row.get(col_map.get("year"), None),
        "venue": row.get(col_map.get("venue"), ""),
        "url": row.get(col_map.get("url"), ""),
        "citation_count": row.get(col_map.get("citation_count"), 0),
        "abstract": row.get(col_map.get("abstract"), ""),
        "tldr": row.get(col_map.get("tldr"), ""),
        "fields_of_study": row.get(col_map.get("fields_of_study"), ""),
        "research_topic": row.get(col_map.get("research_topic"), ""),
        "search_text": row.get("search_text", ""),
    })

assert len(metadata) == vectors.shape[0]


## Persist embeddings + metadata

In [4]:
emb_mod.save_embeddings(vectors, metadata, backend=backend)
print("Saved to:", config.EMBEDDINGS_PATH, "and", config.EMBEDDINGS_METADATA_PATH)


2026-09-01 22:21:27,749 | INFO     | src.embeddings | Saved 583 embeddings (dim=768, backend=sentence-transformers) to E:\Job Base Programe\ResearchMind\data\embeddings\paper_embeddings.npy and metadata to E:\Job Base Programe\ResearchMind\data\embeddings\metadata.json


Saved to: E:\Job Base Programe\ResearchMind\data\embeddings\paper_embeddings.npy and E:\Job Base Programe\ResearchMind\data\embeddings\metadata.json


## Sanity check: round-trip load

In [5]:
loaded_vectors, loaded_records, payload = emb_mod.load_embeddings()
assert loaded_vectors.shape == vectors.shape
assert len(loaded_records) == len(metadata)
print("Backend recorded in metadata:", payload["embedding_backend"])
print("Embedding dim:", payload["embedding_dim"])
print("OK — embeddings and metadata are synchronized.")


Backend recorded in metadata: sentence-transformers
Embedding dim: 768
OK — embeddings and metadata are synchronized.
